## 📊 2. Diagram Ilustrasi Konsep

![Ilustrasi Regresi Logistik & Sigmoid](images/img_07_logistic_regression.png)

```
             +-----------------------------------------------+
             |           FUNGSI SIGMOID / LOGISTIK           |
             +-----------------------------------------------+
             
       P(Y=1) ^
          1.0 |                              .-------- (Probabilitas -> 1)
              |                          . '
          0.5 |-----------------------+ (Threshold Klasifikasi)
              |                   . '
          0.0 | --------. ' ---------------------------> z (Score Logit)
             -inf                    0                    +inf
             
             Rumus:  P(Y=1) = 1 / (1 + exp(-z))
             di mana z = beta_0 + beta_1*X1 + ... + beta_k*Xk
             
             Odds Ratio = exp(beta_i)
             (Kenaikan odds peluang saat X_i naik 1 unit)
```


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
df_credit = pd.read_csv("../datasets/05_credit_risk_classification.csv")
print("Dataset Risiko Kredit UMKM dimuat. Total baris:", len(df_credit))
display(df_credit.head())


## 🧪 3. Pemodelan Regresi Logistik dengan `statsmodels` (Output Logit & MLE)


In [ ]:
features = ['monthly_revenue_million', 'business_experience_yrs', 'has_side_business', 
            'credit_history_good', 'num_dependents']
X = df_credit[features]
y = df_credit['credit_status_smooth']

X_const = sm.add_constant(X)
logit_model = sm.Logit(y, X_const).fit()

print(logit_model.summary())


## 📈 4. Tabel Koefisien, Odds Ratio ($e^eta$), dan Formula Prediksi


In [ ]:
odds_ratios = np.exp(logit_model.params)
conf_int = np.exp(logit_model.conf_int())

logit_summary = pd.DataFrame({
    'Koefisien (β)': logit_model.params,
    'Std Error': logit_model.bse,
    'z-statistic': logit_model.tvalues,
    'p-value': logit_model.pvalues,
    'Odds Ratio (e^β)': odds_ratios,
    '95% CI Lower': conf_int[0],
    '95% CI Upper': conf_int[1]
})

print("=== Tabel Estimasi Regresi Logistik & Odds Ratio ===")
display(logit_summary.round(3))

# Menampilkan formula prediksi
formula_str = f"z = {logit_model.params['const']:.2f}"
for col in features:
    coef = logit_model.params[col]
    sign = "+" if coef >= 0 else "-"
    formula_str += f" {sign} {abs(coef):.2f} * {col}"

print("\n=== Formula Prediksi Probabilitas Kredit Lancar ===")
print(f"Score Logit (z): {formula_str}")
print("P(Kredit Lancar = 1) = 1 / (1 + exp(-z))")


## 🎯 5. Evaluasi Klasifikasi: Confusion Matrix & Kurva ROC-AUC


In [ ]:
y_prob = logit_model.predict(X_const)
y_pred = (y_prob >= 0.5).astype(int)

cm = confusion_matrix(y, y_pred)
auc_val = roc_auc_score(y, y_prob)
fpr, tpr, _ = roc_curve(y, y_prob)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Confusion Matrix
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0], cbar=False,
            xticklabels=['Macet (0)', 'Lancar (1)'], yticklabels=['Macet (0)', 'Lancar (1)'])
axes[0].set_title('Confusion Matrix (Threshold = 0.5)', fontweight='bold')
axes[0].set_xlabel('Prediksi Model')
axes[0].set_ylabel('Status Aktual')

# Subplot 2: Kurva ROC
axes[1].plot(fpr, tpr, color='darkblue', lw=2, label=f'ROC Curve (AUC = {auc_val:.3f})')
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--')
axes[1].set_title('Receiver Operating Characteristic (ROC)', fontweight='bold')
axes[1].set_xlabel('False Positive Rate (1 - Specificity)')
axes[1].set_ylabel('True Positive Rate (Sensitivity)')
axes[1].legend()

plt.tight_layout()
plt.show()

print("=== Laporan Klasifikasi Lengkap ===")
print(classification_report(y, y_pred, target_names=['Kredit Macet (0)', 'Kredit Lancar (1)']))


## 📝 Kesimpulan Analisis

### Q&A
* **Mengapa kasus kelancaran kredit lebih cocok menggunakan regresi logistik dibanding regresi linier?** Karena variabel dependen bersifat kategorik biner (1 = Lancar, 0 = Macet). Regresi linier dapat menghasilkan prediksi di luar rentang probabilitas valid $[0, 1]$, sedangkan fungsi logistik menjamin output berupa probabilitas yang valid.
* **Bagaimana menginterpretasi Odds Ratio riwayat kredit $e^eta = 5.75$?** Pelaku UMKM yang memiliki riwayat kredit baik memiliki peluang (*odds*) kelancaran pembayaran **5,75 kali lebih tinggi** dibandingkan yang pernah bermasalah.

### Data Analysis Key Findings
* Model mencapai nilai **ROC-AUC = {auc_val:.3f}** dan akurasi tinggi, menandakan kemampuan diskriminasi model yang sangat baik dalam memisahkan debitur lancar dan macet.
* Fitur dengan kontribusi positif terbesar adalah `credit_history_good` dan `monthly_revenue_million`, sedangkan jumlah tanggungan keluarga (`num_dependents`) menurunkan peluang kelancaran kredit secara signifikan.

### Insights or Next Steps
* Ambang batas (*threshold*) klasifikasi 0.5 dapat digeser (misal ke 0.6) jika pihak bank ingin menerapkan kebijakan manajemen risiko kredit yang lebih konservatif.
